# scrape judul dan url


In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://seputarmalang.com/pendidikan/"

html = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
).text

soup = BeautifulSoup(html, "html.parser")

data = []

for article in soup.select("article.jeg_post"):

    title_tag = article.select_one(
        "h3.jeg_post_title a"
    )

    if not title_tag:
        continue

    data.append({
        "title": title_tag.get_text(strip=True),
        "url": title_tag["href"],
        "source": "seputar_malang"
    })

df = pd.DataFrame(data).drop_duplicates()

print(df.head())
print(f"Found {len(df)} articles")

df.to_csv(
    "seputar_malang_pendidikan.csv",
    index=False
)

                                               title  \
0  Dorong Sertifikasi Halal di Sekolah dan Madras...   
1  Prestasi Gemilang! SMKN 8 Kota Malang Raih Jua...   
2  Prestasi Membanggakan, 16 Siswa SMKN 8 Kota Ma...   
3  Fapet UB Kupas Tuntas Peluang Karier Industri ...   
4  Unira Malang Hadirkan Menko Zulhas dalam Kulia...   

                                                 url          source  
0  https://seputarmalang.com/pendidikan/seputar-k...  seputar_malang  
1  https://seputarmalang.com/pendidikan/seputar-h...  seputar_malang  
2  https://seputarmalang.com/pendidikan/seputar-h...  seputar_malang  
3  https://seputarmalang.com/pendidikan/seputar-k...  seputar_malang  
4  https://seputarmalang.com/pendidikan/seputar-k...  seputar_malang  
Found 15 articles


# scrape isi url 

In [2]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# hasil dari scrape list artikel sebelumnya
df = pd.read_csv("seputar_malang_pendidikan.csv")

articles = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for i, row in df.iterrows():

    try:

        html = requests.get(
            row["url"],
            headers=headers,
            timeout=30
        ).text

        soup = BeautifulSoup(
            html,
            "html.parser"
        )

        title = (
            soup.select_one("h1.jeg_post_title")
            .get_text(strip=True)
            if soup.select_one("h1.jeg_post_title")
            else None
        )

        subtitle = (
            soup.select_one("h2.jeg_post_subtitle")
            .get_text(strip=True)
            if soup.select_one("h2.jeg_post_subtitle")
            else None
        )

        author = (
            soup.select_one(".jeg_meta_author a")
            .get_text(strip=True)
            if soup.select_one(".jeg_meta_author a")
            else None
        )

        published_date = (
            soup.select_one(".jeg_meta_date a")
            .get_text(strip=True)
            if soup.select_one(".jeg_meta_date a")
            else None
        )

        content_div = soup.select_one(
            "div.content-inner"
        )

        content = None

        if content_div:

            for tag in content_div.select(
                "script,style,figure,.wp-caption,.sharedaddy"
            ):
                tag.decompose()

            content = "\n".join([
                p.get_text(" ", strip=True)
                for p in content_div.select("p")
            ])

        articles.append({
            "title": title,
            "subtitle": subtitle,
            "author": author,
            "published_date": published_date,
            "content": content,
            "url": row["url"]
        })

        print(
            f"[{i+1}/{len(df)}] success"
        )

    except Exception as e:

        print(
            f"[{i+1}/{len(df)}] failed: {e}"
        )

result = pd.DataFrame(articles)

result.to_csv(
    "seputar_malang_articles.csv",
    index=False
)

print(
    f"saved {len(result)} articles"
)

result.head()

[1/15] success
[2/15] success
[3/15] success
[4/15] success
[5/15] success
[6/15] success
[7/15] success
[8/15] success
[9/15] success
[10/15] success
[11/15] success
[12/15] success
[13/15] success
[14/15] success
[15/15] success
saved 15 articles


,title,subtitle,author,published_date,content,url
0,Dorong Sertifikasi Halal di Sekolah dan Madras...,"Mayoritas Muslim, Minim Jaminan Halal? Ini Upa...",Kontributor,19 April 2026,"Malang, SeputarMalang.Com – Indonesia adalah s...",https://seputarmalang.com/pendidikan/seputar-k...
1,Prestasi Gemilang! SMKN 8 Kota Malang Raih Jua...,Siswa SMKN 8 Kota Malang Torehkan Prestasi Gem...,Abe,10 April 2026,"Surabaya, SeputarMalang.Com – Kamis Malam (9/0...",https://seputarmalang.com/pendidikan/seputar-h...
2,"Prestasi Membanggakan, 16 Siswa SMKN 8 Kota Ma...",Paradigma Pendidikan Vokasi di SMK 8 Kota Mala...,Kontributor,3 April 2026,"Malang, SeputarMalang.Com – Sekolah Menegah Ke...",https://seputarmalang.com/pendidikan/seputar-h...
3,Fapet UB Kupas Tuntas Peluang Karier Industri ...,NaN,Kontributor,18 Maret 2026,"Malang, SeputarMalang.Com – Mahasiswa masa kin...",https://seputarmalang.com/pendidikan/seputar-k...
4,Unira Malang Hadirkan Menko Zulhas dalam Kulia...,Menko Zulhas: Bangsa Kuat Dimulai dari Ketahan...,Abe,24 Februari 2026,"Malang, SeputarMalang.Com – Menteri Koordinato...",https://seputarmalang.com/pendidikan/seputar-k...
